# TDSP — train on Colab GPU

Trains the MPNN+DDQN agent on the **total dominating set** problem, multi-size,
with checkpoint/resume so a free-tier disconnect does not lose work.

**Before running:** `Runtime -> Change runtime type -> T4 GPU` (see the guide
the assistant wrote; a GPU is required for the speedup).

Run the cells top to bottom. Everything checkpoints to Google Drive, so you can
close the tab and come back: re-run cells 1-3 and the resumed cell.

## 0. Check the GPU is attached

In [ ]:
!nvidia-smi -L
import torch
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU. Runtime -> Change runtime type -> T4 GPU, then re-run.'

## 1. Mount Google Drive (checkpoints + results survive a disconnect)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
RUN_ROOT = '/content/drive/MyDrive/tdsp_runs'   # <-- everything is written here
os.makedirs(RUN_ROOT, exist_ok=True)
print('checkpoints will live in', RUN_ROOT)

## 2. Get the code

Pick **one** of the two options:

* **Option A (GitHub)** — needs the repo pushed. Adjust `REPO`/`BRANCH` if needed.
* **Option B (upload a zip)** — works without GitHub credentials; upload the
  `dsp_rl_code.zip` the assistant produced.

In [ ]:
USE = 'github'   # 'github' or 'zip'
REPO   = 'https://github.com/mahmoud-yousefi/files.git'
BRANCH = 'tdsp-colab'

if USE == 'github':
    !git clone -b {BRANCH} {REPO} /content/files
else:
    from google.colab import files as _f
    up = _f.upload()
    import zipfile, os
    os.makedirs('/content/files', exist_ok=True)
    name = list(up)[0]
    zipfile.ZipFile(name).extractall('/content/files')

CODE = '/content/files/dsp_rl'
%cd $CODE
!ls

## 3. Install dependencies

In [ ]:
!pip -q install networkx scipy
import torch, networkx, scipy, numpy
print('ok | torch', torch.__version__, '| cuda', torch.cuda.is_available())

## 4. Train

This is the experiment: **multi-size training with the paper's episode budgets**
(`--sizes 20,40,80,100`, 800k episodes) on the corrected architecture.

* `--device cuda` puts the network forward/backward on the GPU.
* `--out $RUN_ROOT/tdsp_ms` writes checkpoint + log straight to Drive.
* Re-running this cell **resumes** from the last checkpoint automatically.

If a run is interrupted: just re-run this cell (it resumes). Nothing is lost.

In [ ]:
OUT = f'{RUN_ROOT}/tdsp_ms'          # one folder for the whole multi-size run
EPISODES = 800000

!python train.py --env tdsp --reward size --sizes 20,40,80,100 \
    --episodes {EPISODES} --seed 0 --device cuda \
    --eval_every 5000 --val_graphs 50 --ckpt_every 10000 \
    --resume --out {OUT}

## 5. Evaluate the trained model (optional, after training finishes)

Sweeps the families/sizes and writes `eval.md` (model vs greedy vs optimum).

In [ ]:
!python experiments/tdsp/exp4_eval.py --model ms={OUT}/model.pt \
    --reps 20 --out {RUN_ROOT}/eval_ms

## 6. (Optional) zip the results back to your laptop

The `tdsp_runs` folder is already on your Drive, so you can also just download
it from Drive. This cell makes a single zip next to the notebook.

In [ ]:
!cp {RUN_ROOT}/tdsp_ms/config.json {RUN_ROOT}/tdsp_ms/log.csv /content/ 2>/dev/null; \
cd /content && zip -qr tdsp_results.zip tdsp_ms_eval tdsp_ms config.json log.csv 2>/dev/null; \
ls -la /content/*.zip
from google.colab import files as _f
_f.download('/content/tdsp_results.zip')